In [7]:
import random as rnd
import numpy as np

In [13]:
class Firm():
    def __init__(self, id: int, country: int, sector: int):
        self.id = id
        self.country = country
        self.sector = sector
    def info(self):
        print(self.id, "c = ", self.country, "s = ",self.sector)

In [14]:
rnd.seed = 12345
#minisetup firm
id = 0
for country in range(26):
    for sector in range(64):
        for n in range(rnd.randint(1,1000)):
            aFirm = Firm(id, country, sector)
            id += 1
print (id)

832304


In [15]:
aFirm.info()

832303 c =  25 s =  63


In [1]:
def match_buyers_to_suppliers(
    buyers,
    suppliers,
    buyer_country: int,
    k: float,
    rng: np.random.Generator,
    point_mode: str = "systematic",
):
    """
    Associa ciascun buyer di una country a un supplier di un settore
    usando due nastri geometricamente affiancati.

    Parameters
    ----------
    buyers
        Sequenza dei buyer appartenenti alla stessa country.

    suppliers
        Sequenza dei supplier appartenenti allo stesso settore.

    buyer_country
        Country dei buyer, compresa tra 0 e 26.

    k
        Moltiplicatore applicato ai supplier della stessa country.

    rng
        Generatore NumPy, per esempio:
        rng = np.random.default_rng(12345)

    point_mode
        "starts":
            usa esattamente l'inizio di ciascun segmento buyer;

        "systematic":
            trasla tutti i punti di una stessa quantità casuale
            inferiore alla lunghezza di un segmento buyer;

        "stratified":
            colloca casualmente il punto all'interno di ciascun
            segmento buyer.

    Returns
    -------
    supplier_uids
        Array contenente, per ogni buyer, lo uid del supplier assegnato.

    supplier_indices
        Posizione del supplier nella sequenza originale suppliers.
    """

    n_buyers = len(buyers)
    n_suppliers = len(suppliers)

    if n_buyers == 0:
        return (
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64),
        )

    if n_suppliers == 0:
        raise ValueError("La lista dei supplier è vuota.")

    if k <= 0:
        raise ValueError("k deve essere strettamente positivo.")

    # Estrazione dei dati una sola volta.
    supplier_uids = np.fromiter(
        (supplier.uid for supplier in suppliers),
        dtype=np.int64,
        count=n_suppliers,
    )

    supplier_countries = np.fromiter(
        (supplier.country for supplier in suppliers),
        dtype=np.int16,
        count=n_suppliers,
    )

    added_values = np.fromiter(
        (supplier.targetAddedValue for supplier in suppliers),
        dtype=np.float64,
        count=n_suppliers,
    )

    # Valori negativi non sono ammissibili.
    if np.any(added_values < 0):
        raise ValueError("targetAddedValue contiene valori negativi.")

    # Costruzione della versione B_s,c del nastro.
    weights = added_values.copy()
    weights[supplier_countries == buyer_country] *= k

    total_weight = weights.sum()

    if not np.isfinite(total_weight) or total_weight <= 0:
        raise ValueError(
            "La somma dei pesi dei supplier deve essere positiva e finita."
        )

    # Shuffle del nastro B.
    permutation = rng.permutation(n_suppliers)

    shuffled_weights = weights[permutation]
    shuffled_uids = supplier_uids[permutation]

    # Estremi destri degli intervalli dei supplier.
    cumulative_weights = np.cumsum(shuffled_weights)

    # Correzione precauzionale dell'ultimo estremo.
    cumulative_weights[-1] = total_weight

    # Posizioni geometriche dei buyer nel nastro [0, total_weight).
    if point_mode == "starts":

        # Inizio esatto di ogni segmento di A.
        buyer_points = (
            np.arange(n_buyers, dtype=np.float64)
            * total_weight
            / n_buyers
        )

    elif point_mode == "systematic":

        # Traslazione casuale comune:
        # conserva punti perfettamente equidistanti.
        offset = rng.random() / n_buyers

        buyer_points = (
            offset + np.arange(n_buyers, dtype=np.float64) / n_buyers
        ) * total_weight

    elif point_mode == "stratified":

        # Un punto casuale entro ciascun segmento di A.
        buyer_points = (
            np.arange(n_buyers, dtype=np.float64) + rng.random(n_buyers)
        ) * total_weight / n_buyers

    else:
        raise ValueError(
            "point_mode deve essere 'starts', "
            "'systematic' oppure 'stratified'."
        )

    # Individuazione dell'intervallo B contenente ciascun punto A.
    shuffled_indices = np.searchsorted(
        cumulative_weights,
        buyer_points,
        side="right",
    )

    # Supplier assegnato a ogni buyer.
    matched_uids = shuffled_uids[shuffled_indices]

    # Indice del supplier nella lista originale.
    original_indices = permutation[shuffled_indices]

    return matched_uids, original_indices

In [2]:
buyers_by_country[c]

suppliers_by_sector[s]

NameError: name 'buyers_by_country' is not defined